# Slow Moving Products View ETL

## Purpose
Identifies products with low sales performance to support inventory optimization and marketing decisions. This view enables:
* Identify underperforming products requiring attention
* Support inventory reduction decisions
* Plan promotional campaigns for slow movers
* Optimize product portfolio management
* Free up capital tied in slow-moving inventory

## Input
* **Source:** `big_data.gold.product_performance` (assumed catalog/schema)

## Output
* **Target:** `big_data.gold.vw_slow_moving_products`
* **Refresh:** Real-time (always reflects current source data)

## SQL Logic
1. Filter products from product_performance table
2. Apply slow-moving criteria:
   * **Times Ordered:** Less than 50 orders (low sales volume)
   * **Reorder Rate:** Less than 20% (customers don't repurchase)
3. Calculate additional metrics:
   * **First Order Only Count:** Products bought once but not repurchased
   * **First Order Rate:** Percentage of non-repeat purchases
   * **Revenue per Order:** Average revenue per transaction
4. Order by severity (worst performers first):
   * Primary: Lowest times_ordered (least sales activity)
   * Secondary: Lowest reorder_rate (worst customer retention)
   * Tertiary: Lowest estimated_revenue_usd (least revenue impact)

In [0]:
%sql
-- Slow Moving Products View
-- Purpose: Identify low-performing products with low order frequency and reorder rates
-- Ordered by severity: least orders, worst retention, lowest revenue

CREATE OR REPLACE VIEW big_data.gold.vw_slow_moving_products AS
SELECT 
  -- Product Identification
  product_id,
  product_name,
  department,
  aisle,
  
  -- Sales Volume Metrics
  times_ordered,                                             -- Total times product was ordered
  times_reordered,                                           -- Times product was reordered (repeat purchase)
  (times_ordered - times_reordered) AS times_first_order,    -- Times bought only once (no repeat)
  
  -- Customer Retention Metrics
  reorder_rate,                                              -- % of customers who repurchase (low = poor retention)
  ROUND(
    ((times_ordered - times_reordered) / times_ordered) * 100, 2
  ) AS first_order_only_rate,                                -- % of one-time purchases (high = no loyalty)
  
  -- Revenue Metrics
  estimated_revenue_usd,                                     -- Total revenue generated
  price_usd,                                                 -- Product price
  ROUND(
    estimated_revenue_usd / times_ordered, 2
  ) AS revenue_per_order                                     -- Average revenue per transaction
  
FROM big_data.gold.product_performance
WHERE times_ordered < 50      -- Low sales volume
  AND reorder_rate < 20       -- Poor customer retention
ORDER BY 
  times_ordered ASC,          -- Least sales activity first (primary)
  reorder_rate ASC,           -- Worst retention second (secondary)
  estimated_revenue_usd ASC;  -- Lowest revenue impact last (tertiary)

In [0]:
%sql
-- Verify view exists and preview results
-- Shows top 20 slowest moving products with all calculated metrics
-- Products are ranked by severity: least orders → worst retention → lowest revenue

SELECT * FROM big_data.gold.vw_slow_moving_products